In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict, Literal
from pydantic import BaseModel, Field

## loading variable

In [2]:
load_dotenv()

True

## define LLM

In [3]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.2, max_output_tokens=2048)

## define structured output schemas

In [4]:
class SentimentSchema(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(description='Sentiment of the review')


class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')

In [5]:
structured_model = model.with_structured_output(SentimentSchema)
structured_model2 = model.with_structured_output(DiagnosisSchema)

## quick test of structured output

In [6]:
prompt = 'Diagnose this negative review: The software is too buggy and support never replies'
structured_model2.invoke(prompt)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


DiagnosisSchema(issue_type='Support', tone='frustrated', urgency='high')

## define state

In [7]:
class ReviewState(TypedDict):
    review: str
    sentiment: Literal["positive", "negative"]
    diagnosis: dict
    response: str

## define nodes

In [8]:
def find_sentiment(state: ReviewState):
    prompt = f'For the following review find out the sentiment \n {state["review"]}'
    sentiment = structured_model.invoke(prompt).sentiment
    return {'sentiment': sentiment}


def positive_response(state: ReviewState):
    prompt = f"""Write a warm thank-you message in response to this review:

"{state['review']}"

Also, kindly ask the user to leave feedback on our website."""
    response = model.invoke(prompt).content
    return {'response': response}


def run_diagnosis(state: ReviewState):
    prompt = f"""Diagnose this negative review:

{state['review']}

Return issue_type, tone, and urgency."""
    response = structured_model2.invoke(prompt)
    return {'diagnosis': response.model_dump()}


def negative_response(state: ReviewState):
    diagnosis = state['diagnosis']
    prompt = f"""You are a support assistant.
The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
Write an empathetic, helpful resolution message.
"""
    response = model.invoke(prompt).content
    return {'response': response}


def check_sentiment(state: ReviewState) -> Literal["positive_response", "run_diagnosis"]:
    if state['sentiment'] == 'positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'

## define and compile graph

In [ ]:
graph = StateGraph(ReviewState)

# add nodes
graph.add_node('find_sentiment', find_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)

# add edges with conditions
# check_sentiment returns "positive_response" or "run_diagnosis" - langgraph routes to
# the node registered under that exact name, so these strings must match the
# add_node(...) names above (and the Literal[...] on check_sentiment)
graph.add_edge(START, 'find_sentiment')
graph.add_conditional_edges('find_sentiment', check_sentiment)

graph.add_edge('positive_response', END)

graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)

workflow = graph.compile()

workflow

## run with example input - positive review

In [10]:
initial_state = {
    'review': "Excellent software with a clean interface and smooth performance. It has improved our workflow significantly and saved a lot of time. Highly recommended for anyone looking for a reliable tech solution!"
}
result = workflow.invoke(initial_state)
result

{'review': 'Excellent software with a clean interface and smooth performance. It has improved our workflow significantly and saved a lot of time. Highly recommended for anyone looking for a reliable tech solution!',
 'sentiment': 'positive',
 'response': 'Wow, thank you so much for this incredibly kind and detailed review!\n\nWe\'re absolutely thrilled to hear that our software\'s clean interface and smooth performance have made such a positive impact on your workflow, significantly improving it and saving you valuable time. It means the world to us that you find it an "excellent" and "reliable tech solution" and "highly recommend" it.\n\nYour feedback is truly invaluable to us, not just for our team, but also for helping others discover how our solution can benefit them. If you have a moment, we would be incredibly grateful if you could share your experience on our website as well.\n\nThanks again for your wonderful support!'}

## run with example input - negative review

In [11]:
initial_state = {
    'review': "The software has potential, but the performance is inconsistent and the interface can be confusing at times. Frequent bugs and slow customer support made the overall experience frustrating. Needs significant improvement."
}
result = workflow.invoke(initial_state)
result

{'review': 'The software has potential, but the performance is inconsistent and the interface can be confusing at times. Frequent bugs and slow customer support made the overall experience frustrating. Needs significant improvement.',
 'sentiment': 'negative',
 'diagnosis': {'issue_type': 'Bug', 'tone': 'frustrated', 'urgency': 'high'},
 'response': 'Subject: Good News! Your High-Priority Bug Has Been Resolved - [Specific Bug Area, e.g., Login Issue, Data Display Error]\n\nHi [User Name],\n\nI\'m writing to you with an update on the high-priority bug you reported, and I want to start by sincerely apologizing for the frustration and inconvenience this issue has caused you. We completely understand how critical this functionality is and how disruptive it must have been, especially given the urgency you highlighted.\n\nOur team immediately escalated your report and has been working diligently to identify and resolve the problem. I\'m very happy to confirm that **we have successfully ident